# Travel-Mode Constraint Function

## Overview

This notebook defines and validates a travel-mode constraint function.

For profiles aged 0–4 or 5–11, `Car or van driver` and `Motorcycle` are removed. Where the car-ownership attribute indicates `No cars or vans in household`, `Car or van driver` is removed for all age groups. For profiles from households without a car or van, `Car or van passenger` is additionally removed. `Taxi or minicab` remains available because it does not depend on household vehicle availability.

## 1. Import Libraries

In [1]:
import json
import sys

from copy import deepcopy
from pathlib import Path

import pandas as pd

In [2]:
project_root = Path.cwd().parents[1]

if str(project_root) not in sys.path:
    sys.path.insert(
        0,
        str(project_root),
    )

## 2. Set Up File Paths

In [3]:
project_root = Path.cwd().parents[1]

representative_profile_space = (
    project_root
    / "output"
    / "representative_profiles"
    / "constrained_representative_profile_space.parquet"
)

behavioural_evidence_layer = (
    project_root
    / "output"
    / "behavioural_evidence"
    / "nts_behavioural_evidence_layer.json"
)

## 3. Load Input Data

In [4]:
profile_space = pd.read_parquet(representative_profile_space)

with open(
    behavioural_evidence_layer,
    "r",
    encoding="utf-8",
) as file:
    behavioural_evidence = json.load(file)

print("Profile space:", profile_space.shape)
print("Trip purposes loaded:", len(behavioural_evidence))

Profile space: (2147608, 16)
Trip purposes loaded: 14


## 4. Define Travel-Mode Constraint Function

The function accepts an age category, a household car-ownership category, and one travel-mode probability distribution.

For profiles aged 0–4 or 5–11, **Car or van driver** and **Motorcycle** are removed. If the household has no car or van, both **Car or van driver** and **Car or van passenger** are removed regardless of age.

After the constraints are applied, the remaining probabilities are renormalised to sum to one. Profiles unaffected by either constraint retain their original distribution.

In [ ]:
# AI Collaboration Record
# Tool: ChatGPT (OpenAI)
# Purpose: Add a study-area-based geographic travel-mode constraint.
# Prompt summary: Retain the original NTS0409a London-specific travel-mode
# categories, but modify the travel-mode constraint and demographic-behavioural
# integration functions so that “Bus in London” and “London Underground” are
# removed for non-London study areas. Renormalise the remaining mode
# probabilities to sum to one and validate their removal for Manchester.

from copy import deepcopy
from ontology.STUDY_AREA import STUDY_AREA


LONDON_SPECIFIC_MODES = {
    "Bus in London",
    "London Underground",
}


def apply_travel_mode_constraint(
    age,
    car_ownership,
    travel_mode_distribution,
    study_area,
):
    constrained = deepcopy(
        travel_mode_distribution
    )

    young_framework_age_groups = {
        "0-4",
        "5-11",
    }

    no_household_car = (
        car_ownership
        == "No cars or vans in household"
    )

    normalised_study_area = (
        study_area.strip().casefold()
    )

    modes_to_remove = set()

    # Remove London-specific modes
    # when the study area is not London.
    if normalised_study_area != "london":
        modes_to_remove.update(
            LONDON_SPECIFIC_MODES
        )

    # Young children cannot drive
    # a car or motorcycle.
    if age in young_framework_age_groups:
        modes_to_remove.update({
            "Car or van driver",
            "Motorcycle",
        })

    # A household without a car cannot
    # support car-based travel.
    if no_household_car:
        modes_to_remove.update({
            "Car or van driver",
            "Car or van passenger",
        })

    # Preserve unaffected distributions.
    if not modes_to_remove:
        return constrained

    for mode in modes_to_remove:
        constrained.pop(mode, None)

    total = sum(constrained.values())

    if total <= 0:
        raise ValueError(
            "No feasible travel-mode "
            "probability remains."
        )

    return {
        mode: probability / total
        for mode, probability
        in constrained.items()
    }

## 5. Select Test Profiles

In [6]:
test_profiles = [
    {
        "Age": "0-4",
        "Car Ownership":
            "1 car or van in household",
        "Expected Removed": {
            "Car or van driver",
            "Motorcycle",
        },
    },
    {
        "Age": "5-11",
        "Car Ownership":
            "1 car or van in household",
        "Expected Removed": {
            "Car or van driver",
            "Motorcycle",
        },
    },
    {
        "Age": "0-4",
        "Car Ownership":
            "No cars or vans in household",
        "Expected Removed": {
            "Car or van driver",
            "Car or van passenger",
            "Motorcycle",
        },
    },
    {
        "Age": "5-11",
        "Car Ownership":
            "No cars or vans in household",
        "Expected Removed": {
            "Car or van driver",
            "Car or van passenger",
            "Motorcycle",
        },
    },
    {
        "Age": "23-34",
        "Car Ownership":
            "No cars or vans in household",
        "Expected Removed": {
            "Car or van driver",
            "Car or van passenger",
        },
    },
    {
        "Age": "23-34",
        "Car Ownership":
            "1 car or van in household",
        "Expected Removed": set(),
    },
]

profile_columns = [
    "Age",
    "Sex",
    "Household Category",
    "Economic Activity Status",
    "Industry",
    "Ethnic Group",
    "Distance travelled to work",
    "Car Ownership",
]

original_test_profiles = []

for case in original_test_profiles:

    candidates = profile_space[
        (
            profile_space["Age"]
            == case["Age"]
        )
        & (
            profile_space["Car Ownership"]
            == case["Car Ownership"]
        )
    ]

    if candidates.empty:
        raise ValueError(
            "No feasible profile found for "
            f"age={case['Age']} and "
            "car ownership="
            f"{case['Car Ownership']}."
        )

    profile = (
        candidates
        .iloc[0][profile_columns]
        .to_dict()
    )

    profile["Expected Removed"] = (
        case["Expected Removed"]
    )

    test_profiles.append(profile)

test_profiles_df = pd.DataFrame([
    {
        **profile,
        "Expected Removed": (
            ", ".join(sorted(
                profile["Expected Removed"]
            ))
            if profile["Expected Removed"]
            else "None"
        ),
    }
    for profile in test_profiles
])

display(test_profiles_df)

,Age,Car Ownership,Expected Removed
0,0-4,1 car or van in household,"Car or van driver, Motorcycle"
1,5-11,1 car or van in household,"Car or van driver, Motorcycle"
2,0-4,No cars or vans in household,"Car or van driver, Car or van passenger, Motor..."
3,5-11,No cars or vans in household,"Car or van driver, Car or van passenger, Motor..."
4,23-34,No cars or vans in household,"Car or van driver, Car or van passenger"
5,23-34,1 car or van in household,None


## 6. Test and Validate Function

In [7]:
# AI Collaboration Record
# Tool: ChatGPT (OpenAI)
# Purpose: Update travel-mode constraint validation.
# Prompt summary: Include London-specific modes in the expected removals
# for non-London study areas and verify that the actual removals match
# expectations and the remaining probabilities sum to one.

validation_summary = []

for profile in test_profiles:

    age = profile["Age"]

    car_ownership = (
        profile["Car Ownership"]
    )

    # Expected demographic removals
    expected_removed = set(
        profile["Expected Removed"]
    )

    # Expected geographic removals
    london_specific_modes = {
        "Bus in London",
        "London Underground",
    }

    london_study_areas = {
        "london",
        "greater london",
        "city of london",
    }

    normalised_study_area = (
        STUDY_AREA.strip().casefold()
    )

    if (
        normalised_study_area
        not in london_study_areas
    ):
        expected_removed.update(
            london_specific_modes
        )

    mode_sets = []

    for purpose, evidence in (
        behavioural_evidence.items()
    ):

        record = (
            evidence[
                "nts0409a_travel_mode_distribution"
            ][0]
        )

        original = {
            mode: probability
            for mode, probability
            in record.items()
            if mode != "Purpose"
        }

        constrained = (
            apply_travel_mode_constraint(
                age=age,
                car_ownership=car_ownership,
                travel_mode_distribution=(
                    original
                ),
                study_area=STUDY_AREA
            )
        )

        original_modes = set(original)
        remaining_modes = set(constrained)

        removed_modes = (
            original_modes
            - remaining_modes
        )

        # Validate the expected modes
        assert removed_modes == (
            expected_removed
        ), (
            f"Unexpected modes removed for "
            f"age={age}, "
            f"car ownership={car_ownership}, "
            f"purpose={purpose}: "
            f"{removed_modes}"
        )

        # Validate probability sum
        assert abs(
            sum(constrained.values()) - 1
        ) < 1e-8

        # Taxi must always remain available
        assert (
            "Taxi or minicab"
            in constrained
        )

        # Unaffected distributions must
        # remain exactly unchanged
        if not expected_removed:
            assert constrained == original

        mode_sets.append(
            remaining_modes
        )

    # All purposes should retain the
    # same set of permitted modes
    first_mode_set = mode_sets[0]

    assert all(
        modes == first_mode_set
        for modes in mode_sets
    )

    validation_summary.append({
        "Age": age,
        "Car Ownership": car_ownership,
        "Travel Mode Count": len(
            first_mode_set
        ),
        "Removed Travel Mode": (
            ", ".join(sorted(
                expected_removed
            ))
            if expected_removed
            else "None"
        ),
        "Remaining Travel Modes": (
            ", ".join(sorted(
                first_mode_set
            ))
        ),
        "Probability Validation": "PASS",
        "Unchanged Validation": (
            "PASS"
            if not expected_removed
            else "No Unchanged Validation"
        ),
    })

validation_summary = pd.DataFrame(
    validation_summary
)

display(validation_summary)

,Age,Car Ownership,Travel Mode Count,Removed Travel Mode,Remaining Travel Modes,Probability Validation,Unchanged Validation
0,0-4,1 car or van in household,9,"Bus in London, Car or van driver, London Under...","Car or van passenger, Non-local bus, Other Pri...",PASS,No Unchanged Validation
1,5-11,1 car or van in household,9,"Bus in London, Car or van driver, London Under...","Car or van passenger, Non-local bus, Other Pri...",PASS,No Unchanged Validation
2,0-4,No cars or vans in household,8,"Bus in London, Car or van driver, Car or van p...","Non-local bus, Other Private Transport, Other ...",PASS,No Unchanged Validation
3,5-11,No cars or vans in household,8,"Bus in London, Car or van driver, Car or van p...","Non-local bus, Other Private Transport, Other ...",PASS,No Unchanged Validation
4,23-34,No cars or vans in household,9,"Bus in London, Car or van driver, Car or van p...","Motorcycle, Non-local bus, Other Private Trans...",PASS,No Unchanged Validation
5,23-34,1 car or van in household,11,"Bus in London, London Underground","Car or van driver, Car or van passenger, Motor...",PASS,No Unchanged Validation
